# Redrob Track 1 - Sandbox

In [ ]:
!pip install -q rank-bm25 pandas numpy tqdm openpyxl

In [ ]:
import os, json, re, urllib.request
from datetime import datetime
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi

sample_url = "https://raw.githubusercontent.com/AdityaSonune369/India-Runs-Hackathon-Submission/main/data/sample_candidates.json"
sample_path = "sample_candidates.json"
if not os.path.exists(sample_path):
    urllib.request.urlretrieve(sample_url, sample_path)
with open(sample_path, "r", encoding="utf-8") as f:
    candidates = json.load(f)


In [3]:
JD_TEXT = """Senior AI Engineer — Founding Team at Redrob AI (Series A AI-native talent intelligence platform).
Location: Pune/Noida, India (Hybrid). Open to relocation from Tier-1 Indian cities.
Experience: roughly 5-9 years with production depth.

We need someone comfortable with BOTH:
- Deep technical depth in modern ML systems — embeddings, retrieval, ranking, LLMs, fine-tuning.
- Scrappy product-engineering attitude — willing to ship a working ranker in a week even if suboptimal, learn from real users.

Mandate: own the intelligence layer (ranking, retrieval, and matching systems for recruiters and candidates).

Must have:
- Production experience with embeddings-based retrieval systems (sentence-transformers, BGE, E5, etc.) deployed to users.
- Production experience with vector databases or hybrid search (Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, Elasticsearch, FAISS...).
- Strong Python and code quality.
- Hands-on experience designing evaluation frameworks for ranking systems (NDCG, MAP, offline-to-online correlation, A/B).

Hackathon note: The right answer is NOT "candidates whose skills section contains the most AI keywords". That is a trap.
Reason about the gap between what the JD says and what it means. Heavily weigh behavioral signals.
"""
JD_TOKENS = [w.lower() for w in re.findall(r"\w+", JD_TEXT.lower())]
JD_KEY_PHRASES = ["embeddings", "retrieval", "ranking", "vector", "hybrid search", "faiss", "pinecone", "weaviate", "recommendation", "search system", "matching", "candidate", "recruiter", "production", "fine-tuning", "llm", "rag", "evaluation", "ndcg", "map", "a/b", "offline", "online", "ship", "deployed", "users", "scale", "pipeline", "python", "data", "ml", "ai engineer"]
CONSULTING_FLAGS = ["tcs", "infosys", "wipro", "accenture", "cognizant", "capgemini"]

def get_candidate_text(cand):
    parts = []
    prof = cand.get("profile", {})
    parts.append(prof.get("headline", ""))
    parts.append(prof.get("summary", ""))
    for ch in cand.get("career_history", []):
        parts.append(ch.get("title", ""))
        parts.append(ch.get("description", ""))
        parts.append(ch.get("company", ""))
    for sk in cand.get("skills", []):
        parts.append(sk.get("name", ""))
    for edu in cand.get("education", []):
        parts.append(edu.get("institution", ""))
        parts.append(edu.get("field_of_study", ""))
        parts.append(edu.get("degree", ""))
    return " ".join(parts)

def compute_behavioral_score(cand):
    sig = cand.get("redrob_signals", {})
    if not sig: return 0.5
    score = 0.0
    if sig.get("open_to_work_flag"): score += 0.18
    rr = float(sig.get("recruiter_response_rate", 0.0) or 0.0)
    score += rr * 0.22
    views = min(sig.get("profile_views_received_30d", 0) or 0, 50) / 50.0
    score += views * 0.10
    saved = min(sig.get("saved_by_recruiters_30d", 0) or 0, 20) / 20.0
    score += saved * 0.08
    search_app = min(sig.get("search_appearance_30d", 0) or 0, 30) / 30.0
    score += search_app * 0.08
    last_active = sig.get("last_active_date")
    if last_active:
        try:
            days = (datetime.now() - datetime.fromisoformat(last_active)).days
            recency = max(0, 1 - min(days, 90) / 90.0)
            score += recency * 0.10
        except: pass
    completeness = (sig.get("profile_completeness_score", 50) or 50) / 100.0
    score += completeness * 0.06
    gh = sig.get("github_activity_score", -1) or -1
    if gh > 0: score += min(gh, 60) / 120.0
    icr = float(sig.get("interview_completion_rate", 0.0) or 0.0)
    score += icr * 0.08
    apps = sig.get("applications_submitted_30d", 0) or 0
    if apps == 0 and not sig.get("open_to_work_flag"): score -= 0.05
    return float(np.clip(score, 0.0, 1.0))

def compute_jd_overlap_score(cand):
    text_lower = get_candidate_text(cand).lower()
    skills = cand.get("skills", [])
    phrase_hits = sum(1 for p in JD_KEY_PHRASES if p in text_lower)
    phrase_score = min(phrase_hits / 6.0, 1.0) * 0.55
    skill_names_lower = [s.get("name", "").lower() for s in skills]
    skill_hits = sum(1 for p in JD_KEY_PHRASES if any(p in n for n in skill_names_lower))
    skill_score = min(skill_hits / 5.0, 1.0) * 0.25
    prof_bonus = 0.0
    for s in skills:
        if any(kw in s.get("name", "").lower() for kw in ["embed", "retriev", "rank", "vector", "llm", "fine", "python", "ml"]):
            prof = s.get("proficiency", "beginner")
            end = s.get("endorsements", 0) or 0
            if prof in ("advanced", "expert"):
                prof_bonus += 0.03 * min(end / 20.0, 1.0)
    prof_bonus = min(prof_bonus, 0.20)
    return float(np.clip(phrase_score + skill_score + prof_bonus, 0.0, 1.0))

def compute_career_fit(cand):
    career = cand.get("career_history", [])
    if not career: return 0.4
    score = 0.5
    text = " ".join([ch.get("description", "") + " " + ch.get("title", "") for ch in career]).lower()
    build_signals = ["built", "shipped", "deployed", "production", "pipeline", "system", "retrieval", "ranking", "search", "recommend"]
    build_count = sum(text.count(w) for w in build_signals)
    score += min(build_count / 8.0, 0.25)
    consulting_years = 0
    for ch in career:
        if any(c in ch.get("company", "").lower() for c in CONSULTING_FLAGS):
            consulting_years += ch.get("duration_months", 0) or 0
    if consulting_years > 36: score -= 0.25
    if "product" in text or "users" in text or "scale" in text: score += 0.10
    return float(np.clip(score, 0.1, 1.0))

def compute_exp_fit(cand):
    yrs = cand.get("profile", {}).get("years_of_experience", 0) or 0
    if 5 <= yrs <= 9: return 1.0
    elif 4 <= yrs < 5 or 9 < yrs <= 11: return 0.85
    elif 3 <= yrs < 4 or 11 < yrs <= 13: return 0.65
    else: return max(0.3, 1.0 - abs(yrs - 7) * 0.08)

def compute_overall_score(cand, bm25_score, max_bm25):
    norm_bm25 = bm25_score / max(max_bm25, 1e-6)
    beh = compute_behavioral_score(cand)
    overlap = compute_jd_overlap_score(cand)
    career = compute_career_fit(cand)
    exp = compute_exp_fit(cand)
    final = (0.28 * norm_bm25 + 0.32 * beh + 0.18 * overlap + 0.12 * career + 0.10 * exp)
    return float(np.clip(final, 0.0, 1.0))

def generate_reasoning(cand, rank, score):
    prof = cand.get("profile", {})
    sig = cand.get("redrob_signals", {})
    yrs = prof.get("years_of_experience", 0)
    title = prof.get("current_title", "Professional")
    skills = [s["name"] for s in cand.get("skills", [])[:5]]
    rr = sig.get("recruiter_response_rate", 0.0)
    base = f"{title} with {yrs:.1f} yrs; "
    if skills: base += f"key skills include {', '.join(skills[:3])}; "
    signals = []
    if sig.get("open_to_work_flag"): signals.append("open to work")
    if rr > 0.6: signals.append(f"high recruiter response {rr:.0%}")
    if sig.get("profile_views_received_30d", 0) > 8: signals.append("strong recent visibility")
    if sig.get("github_activity_score", -1) > 30: signals.append("github activity")
    if signals: base += "; ".join(signals) + "."
    notice = sig.get("notice_period_days", 0)
    if notice > 60 and rank < 30: base += f" Notice period {notice} days is a concern."
    if rank > 70: base += " Adjacent experience; lower priority vs stronger production retrieval matches."
    return base[:280]

print("Building BM25...")
corpus = []
for c in candidates:
    corpus.append([w.lower() for w in re.findall(r"\w+", get_candidate_text(c))])
bm25 = BM25Okapi(corpus)
bm25_scores = bm25.get_scores(JD_TOKENS)
max_bm25 = float(np.max(bm25_scores)) or 1.0

print("Scoring...")
scored = []
for i, cand in enumerate(candidates):
    s = compute_overall_score(cand, bm25_scores[i], max_bm25)
    scored.append((s, cand))
scored.sort(key=lambda x: x[0], reverse=True)

TOPK = 20
rows = []
for rank, (raw_score, cand) in enumerate(scored[:TOPK], 1):
    score = round(0.99 - (rank - 1) * (0.99 - 0.40) / (TOPK - 1), 4)
    reason = generate_reasoning(cand, rank, score)
    rows.append({"candidate_id": cand["candidate_id"], "rank": rank, "score": score, "reasoning": reason})

df = pd.DataFrame(rows)
print(df)


candidate_id  rank  score                                                                                                                                                                                      reasoning
CAND_0000031     1 0.9900                      Recommendation Systems Engineer with 6.0 yrs; key skills include Go, MLflow, FAISS; open to work; high recruiter response 91%; strong recent visibility; github activity.
CAND_0000001     2 0.9589                                                                 Backend Engineer with 6.9 yrs; key skills include Tailwind, NLP, Image Classification; open to work; strong recent visibility.
CAND_0000016     3 0.9279                              Accountant with 5.3 yrs; key skills include Node.js, Figma, Data Pipelines; open to work; high recruiter response 64%; strong recent visibility; github activity.
CAND_0000038     4 0.8968                          Java Developer with 6.7 yrs; key skills include Kubeflow, Django, Redux; open to 

In [ ]:
from google.colab import files
xlsx_name = "sandbox_submission.xlsx"
df.to_excel(xlsx_name, index=False)
files.download(xlsx_name)
